In [ ]:
import torch
import torchvision.models as models

# 1. Load your trained model (must be MobileNetV3)
model = models.mobilenet_v3_small(pretrained=True) 
model.eval()

# 2. Configure the quantization strategy
model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
model_fp32_prepared = torch.quantization.prepare(model)

# 3. Calibration (normally you'd run a small batch of training data through it, simplified here)
# model_fp32_prepared(dummy_input) 

# 4. Convert to the quantized model
model_int8 = torch.quantization.convert(model_fp32_prepared)

# 5. Export to ONNX (ONNX has good support for quantization)
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(model_int8, dummy_input, "leaf_model_int8.onnx", opset_version=13)

### Increase virtual memory:
sudo nano /etc/dphys-swapfile

Change CONF_SWAPSIZE from 100 to 1024
sudo /etc/init.d/dphys-swapfile restart

pip install onnxruntime

### How to intergrate on Raspberry Pi?

1, OS choice: strongly recommended to use Raspberry Pi OS Lite (32-bit). It has no GUI, leaving all the precious memory for the AI model.

2, Install core dependencies: after connecting via SSH, run the following commands to install the base tools:

sudo apt-get update

sudo apt-get install -y python3-pip python3-pil python3-numpy libatlas-base-dev

3, Install the inference engine (ONNX Runtime): for Pi Zero (ARMv6 architecture), a plain pip install onnxruntime may fail. It's recommended to install a version specifically optimized for Raspberry Pi:

pip3 install onnxruntime

### File transfer and organization
It's recommended to set up a clean project structure on the Raspberry Pi:

/home/pi/leaf_checker/

├── model.onnx          # your quantized model

├── labels.txt          # class names (one per line, e.g. Healthy, Rust, Scab)

├── main.py             # the run script

└── test_images/        # test image folder

You can also use the scp command (run from your computer's terminal) to transfer the model over: scp leaf_model_int8.onnx pi@raspberrypi.local:/home/pi/leaf_checker/

### Writing an efficient inference script
For maximum performance, we use PIL directly for image processing, skipping the heavier OpenCV.

In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image
import time

# 1. Load the model
print("Loading model...")
session = ort.InferenceSession("model.onnx")

# 2. Load labels
with open("labels.txt", "r") as f:
    labels = [line.strip() for line in f.readlines()]

def predict(img_path):
    start_time = time.time()
    
    # 3. Image preprocessing (must exactly match training)
    img = Image.open(img_path).convert('RGB')
    img = img.resize((224, 224))
    
    # Convert to NCHW format and normalize
    img_data = np.array(img).transpose(2, 0, 1).astype(np.float32)
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    img_data = (img_data / 255.0 - mean) / std
    img_data = np.expand_dims(img_data, axis=0)

    # 4. Run inference
    input_name = session.get_inputs()[0].name
    outputs = session.run(None, {input_name: img_data})
    
    # 5. Post-process
    probs = outputs[0]
    idx = np.argmax(probs)
    
    end_time = time.time()
    print(f"Inference time: {end_time - start_time:.2f} sec")
    return labels[idx], probs[0][idx]

# Test
label, score = predict("test.jpg")
print(f"Detection result: {label} (confidence: {score:.2f})")

### Hardware integration suggestions
Since this is about recognizing leaves, you'll probably need to use it together with a Raspberry Pi Camera.

Enable the camera: run sudo raspi-config, and enable Legacy Camera support under Interface Options.

Capture-and-recognize script: you can modify main.py to add photo-trigger logic:

In [ ]:
from subprocess import call

def capture_and_predict():
    # Call the system command to take a photo
    call(["libcamera-still", "-o", "live.jpg", "--immediate"])
    return predict("live.jpg")

Does MobileNetV3 Small need to be trained locally? I  import the pretrained model from the library, and then what?

I want to explain the model's output on the Raspberry Pi using an XAI method, like LRP or SHAP — how should I do that?